In [ ]:
# ⚠️  Do NOT commit real API keys.
# Set these in a .env file (loaded below) or as environment variables.
#
# OPENAI_API_KEY=sk-...
# SERPER_API_KEY=...
# VECTADB_URL=http://localhost:8080  (optional, defaults to localhost)

In [ ]:
# crewai>=0.80 is required by vectadb-crewai.
# crewai[tools] bundles crewai-tools (ScrapeWebsiteTool, SerperDevTool, etc.)
!pip install 'crewai[tools]>=0.80.0' vectadb-crewai python-dotenv httpx

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from crewai import Agent, Task, Crew
import os
from dotenv import load_dotenv, find_dotenv



In [ ]:
# VectaDB integration — audit-traces every LLM call, tool call,
# agent action, and task lifecycle event in real-time.
from vectadb_crewai import VectaDBTracer, VectaDBClient
import os

VECTADB_URL = os.getenv("VECTADB_URL", "http://localhost:8080")

# Create the tracer — fail_silently=True means the crew keeps running
# even if VectaDB isn't reachable, so the demo works offline too.
tracer = VectaDBTracer(
    vectadb_url=VECTADB_URL,
    crew_name="customer_support_crew",
    fail_silently=True,
)
print(f"VectaDB URL: {VECTADB_URL}")
print(f"VectaDB reachable: {tracer._client.is_healthy()}")

In [ ]:
# Aux functions
def load_env():
    _ = load_dotenv(find_dotenv())

def get_openai_api_key():
    load_env()
    openai_api_key = os.getenv("OPENAI_API_KEY")
    return openai_api_key

def get_serper_api_key():
    load_env()
    openai_api_key = os.getenv("SERPER_API_KEY")
    return openai_api_key

# Getting the API Keys
openai_api_key = get_openai_api_key()

In [ ]:
# Model and API keys — read from environment / .env file
os.environ["OPENAI_MODEL_NAME"] = os.getenv("OPENAI_MODEL_NAME", "gpt-3.5-turbo")
# openai_api_key is already loaded by load_env() above; no need to hard-code it here.

## Role Playing, Focus and Cooperation

In [ ]:
support_agent = Agent(
    role="Senior Support Representative",
	goal="Be the most friendly and helpful "
        "support representative in your team",
	backstory=(
		"You work at crewAI (https://crewai.com) and "
        " are now working on providing "
		"support to {customer}, a super important customer "
        " for your company."
		"You need to make sure that you provide the best support!"
		"Make sure to provide full complete answers, "
        " and make no assumptions."
	),
	allow_delegation=False,
	verbose=True
)

- By not setting `allow_delegation=False`, `allow_delegation` takes its default value of being `True`.
- This means the agent _can_ delegate its work to another agent which is better suited to do a particular task.

In [ ]:
support_quality_assurance_agent = Agent(
	role="Support Quality Assurance Specialist",
	goal="Get recognition for providing the "
    "best support quality assurance in your team",
	backstory=(
		"You work at crewAI (https://crewai.com) and "
        "are now working with your team "
		"on a request from {customer} ensuring that "
        "the support representative is "
		"providing the best support possible.\n"
		"You need to make sure that the support representative "
        "is providing full"
		"complete answers, and make no assumptions."
	),
	verbose=True
)

* **Role Playing**: Both agents have been given a role, goal and backstory.
* **Focus**: Both agents have been prompted to get into the character of the roles they are playing.
* **Cooperation**: Support Quality Assurance Agent can delegate work back to the Support Agent, allowing for these agents to work together.

## Tools, Guardrails and Memory

### Tools

- Import CrewAI tools

In [ ]:
from crewai_tools import SerperDevTool, \
                         ScrapeWebsiteTool, \
                         WebsiteSearchTool

### Possible Custom Tools
- Load customer data
- Tap into previous conversations
- Load data from a CRM
- Checking existing bug reports
- Checking existing feature requests
- Checking ongoing tickets
- ... and more

- Some ways of using CrewAI tools.

```Python
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()
```

- Instantiate a document scraper tool.
- The tool will scrape a page (only 1 URL) of the CrewAI documentation.

In [ ]:
docs_scrape_tool = ScrapeWebsiteTool(
    website_url="https://docs.crewai.com/how-to/Creating-a-Crew-and-kick-it-off/"
)

##### Different Ways to Give Agents Tools

- Agent Level: The Agent can use the Tool(s) on any Task it performs.
- Task Level: The Agent will only use the Tool(s) when performing that specific Task.

**Note**: Task Tools override the Agent Tools.

### Creating Tasks
- You are passing the Tool on the Task Level.

In [ ]:
inquiry_resolution = Task(
    description=(
        "{customer} just reached out with a super important ask:\n"
	    "{inquiry}\n\n"
        "{person} from {customer} is the one that reached out. "
		"Make sure to use everything you know "
        "to provide the best support possible."
		"You must strive to provide a complete "
        "and accurate response to the customer's inquiry."
    ),
    expected_output=(
	    "A detailed, informative response to the "
        "customer's inquiry that addresses "
        "all aspects of their question.\n"
        "The response should include references "
        "to everything you used to find the answer, "
        "including external data or solutions. "
        "Ensure the answer is complete, "
		"leaving no questions unanswered, and maintain a helpful and friendly "
		"tone throughout."
    ),
	tools=[docs_scrape_tool],
    agent=support_agent,
)

- `quality_assurance_review` is not using any Tool(s)
- Here the QA Agent will only review the work of the Support Agent

In [ ]:
quality_assurance_review = Task(
    description=(
        "Review the response drafted by the Senior Support Representative for {customer}'s inquiry. "
        "Ensure that the answer is comprehensive, accurate, and adheres to the "
		"high-quality standards expected for customer support.\n"
        "Verify that all parts of the customer's inquiry "
        "have been addressed "
		"thoroughly, with a helpful and friendly tone.\n"
        "Check for references and sources used to "
        " find the information, "
		"ensuring the response is well-supported and "
        "leaves no questions unanswered."
    ),
    expected_output=(
        "A final, detailed, and informative response "
        "ready to be sent to the customer.\n"
        "This response should fully address the "
        "customer's inquiry, incorporating all "
		"relevant feedback and improvements.\n"
		"Don't be too formal, we are a chill and cool company "
	    "but maintain a professional and friendly tone throughout."
    ),
    agent=support_quality_assurance_agent,
)


### Creating the Crew

#### Memory
- Setting `memory=True` when putting the crew together enables Memory.

In [ ]:
crew = Crew(
  agents=[support_agent, support_quality_assurance_agent],
  tasks=[inquiry_resolution, quality_assurance_review],
  verbose=True,
  memory=True
)

### Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

#### Guardrails
- By running the execution below, you can see that the agents and the responses are within the scope of what we expect from them.

In [ ]:
inputs = {
    "customer": "DeepLearningAI",
    "person": "Andrew Ng",
    "inquiry": (
        "I need help with setting up a Crew "
        "and kicking it off, specifically "
        "how can I add memory to my crew? "
        "Can you provide guidance?"
    )
}

# ── VectaDB integration: drop-in replacement for crew.kickoff() ──
# tracer.kickoff() instruments the crew, runs it, then flushes all
# captured events (LLM calls, tool calls, agent steps, task lifecycle)
# to VectaDB as a provenance-linked audit trail.
result = tracer.kickoff(crew, inputs=inputs)

print(f"\n✅ Crew finished. Session ID: {tracer.session_id}")

- Display the final result as Markdown.

In [ ]:
from IPython.display import Markdown
Markdown(result)

Dear Andrew Ng from DeepLearningAI,

Thank you for your inquiry about setting up a Crew and adding memory to it in CrewAI. To add memory to your Crew, please follow the detailed steps below:

1. Log in to your CrewAI account using your credentials.
2. Once logged in, navigate to the "Settings" or "Configuration" section of the platform.
3. Look for the option to manage memory or resources for your Crew.
4. Select the Crew for which you want to add memory.
5. Locate the memory allocation settings for the selected Crew.
6. Increase the memory allocation as needed to enhance your Crew's performance.
7. Save the changes to apply the increased memory allocation to the Crew.

By following these steps, you can effectively add memory to your Crew in CrewAI, allowing it to store and access information, knowledge, and past interactions for improved performance. Utilizing the Pydantic library to define structured outputs and store data in memory will enable your Crew to learn from previous interactions and make more informed decisions in the future. Additionally, managing memory and knowledge within your Crew by defining guardrails, callbacks, and triggers will help control how information is stored and accessed.

If you have any further questions or need additional assistance, please don't hesitate to reach out. We are here to support you in maximizing your CrewAI experience.

Best regards,
[Your Name]
Senior Support Representative
crewAI

---
## VectaDB Audit Trail

The cells below query VectaDB to inspect the full provenance record of the crew run above.  
Each LLM call, tool call, agent action, and task lifecycle event has been persisted as a typed entity  
in the hybrid vector + graph store, enabling both structured lineage queries and semantic similarity search.  

> **Requires VectaDB running locally** (`docker-compose up` in the repo root).  
> If VectaDB is offline the crew run above still completes — audit cells will report no data.

In [ ]:
# ── VectaDB Audit Trail ─────────────────────────────────────────────────
# Query the full provenance trail for this session.
import asyncio, httpx
from IPython.display import display, HTML

async def fetch_session_events(session_id: str):
    async with httpx.AsyncClient(base_url=VECTADB_URL, timeout=10) as http:
        try:
            resp = await http.get("/api/v1/events", params={"session_id": session_id})
            if resp.is_success:
                return resp.json().get("events", [])
        except Exception as e:
            print(f"Could not reach VectaDB: {e}")
    return []

if tracer.session_id:
    events = asyncio.run(fetch_session_events(tracer.session_id))

    from collections import Counter
    counts = Counter(e.get("event_type", "unknown") for e in events)
    print(f"Session: {tracer.session_id}")
    print(f"Total events captured: {len(events)}")
    print()
    print("Event breakdown:")
    for etype, n in sorted(counts.items()):
        print(f"  {etype:<25} {n}")
else:
    print("No session recorded — VectaDB may not have been reachable.")

In [ ]:
# ── Semantic Incident Search ─────────────────────────────────────────────
# VectaDB stores every event with a vector embedding, so you can run
# semantic queries across your entire audit trail.
#
# Example: find any tool calls or agent steps related to 'memory' in this run.

async def semantic_search(query: str, session_id: str, limit: int = 5):
    async with httpx.AsyncClient(base_url=VECTADB_URL, timeout=10) as http:
        try:
            resp = await http.post(
                "/api/v1/query/hybrid",
                json={
                    "vector_query": query,
                    "session_id": session_id,
                    "limit": limit,
                }
            )
            if resp.is_success:
                return resp.json().get("results", [])
        except Exception as e:
            print(f"Could not reach VectaDB: {e}")
    return []

if tracer.session_id:
    results = asyncio.run(
        semantic_search("add memory to crew", tracer.session_id)
    )
    print(f"Semantic search results for 'add memory to crew':")
    if results:
        for r in results:
            print(f"  [{r.get('event_type','?')}] score={r.get('score', 0):.3f}")
            props = r.get('properties', {})
            snippet = props.get('output') or props.get('prompt') or props.get('step_output') or ''
            print(f"    {str(snippet)[:120]}")
    else:
        print("  No results (VectaDB may not be running, or no matching events).")